In [1]:
import numpy as np
import pandas as pd
import pyomo.environ as pyo
import importlib.resources
import sys
import os
import json
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from Customized_Determinstic_PT.utils import read_gmlc_gen
import matplotlib.pyplot as plt

# RTS GMLC

## Read gen.csv

In [13]:
# path for the generators
csv_path = os.path.join(os.getcwd(), '..', 'Data', 'gen.csv')
df = read_gmlc_gen(csv_path)

# path for the cost curves of fossil generators
# the calculation for the cost curve is in dispatches/improve_pt branch
json_path = os.path.join(os.getcwd(), '..', 'Data', 'rtsgmle_gen_linear_cost_curve.json')
with open(json_path, 'rb') as f:
    gen_param_dict = json.load(f)

# path for the buses
bus_path = os.path.join(os.getcwd(), "..", "Data", "bus.csv")
df_bus = pd.read_csv(bus_path)

In [15]:
# extract the CT, CC, STEAM, WIND, PV units
target_fossil_gen = []
target_renew_gen = []
target_special_gen = []
for idx, row in df.iterrows():
    if row['Unit Type'] in ['CT', 'CC', 'STEAM']:
        target_fossil_gen.append(row)
    if row['Unit Type'] in ['PV', 'WIND', 'HYDRO', 'RTPV']:
        target_renew_gen.append(row)
    if row['Unit Type'] in ['CSP', 'NUCLEAR']:
        target_special_gen.append(row)
df_fossil = pd.DataFrame(target_fossil_gen)
df_renew = pd.DataFrame(target_renew_gen)
df_special = pd.DataFrame(target_special_gen)

In [16]:
df_fossil.head()

,GEN UID,Bus ID,Gen ID,Unit Group,Unit Type,Category,Fuel,MW Inj,MVAR Inj,V Setpoint p.u.,...,Emissions N2O Lbs/MMBTU,Emissions CO Lbs/MMBTU,Emissions VOCs Lbs/MMBTU,Damping Ratio,Inertia MJ/MW,Base MVA,Transformer X p.u.,Unit X p.u.,Pump Load MW,Storage Roundtrip Efficiency
0,101_CT_1,101,1,U20,CT,Oil CT,Oil,8.0,4.96,1.0468,...,0.004,0.11,0.040,0,2.8,24.0,0.13,0.32,0,0
1,101_CT_2,101,2,U20,CT,Oil CT,Oil,8.0,4.96,1.0468,...,0.004,0.11,0.040,0,2.8,24.0,0.13,0.32,0,0
2,101_STEAM_3,101,3,U76,STEAM,Coal,Coal,76.0,0.14,1.0468,...,0.004,0.02,0.003,0,3.0,89.0,0.13,0.30,0,0
3,101_STEAM_4,101,4,U76,STEAM,Coal,Coal,76.0,0.14,1.0468,...,0.004,0.02,0.003,0,3.0,89.0,0.13,0.30,0,0
4,102_CT_1,102,1,U20,CT,Oil CT,Oil,8.0,4.88,1.0467,...,0.004,0.11,0.040,0,2.8,24.0,0.13,0.32,0,0


In [17]:
df_renew.head()

,GEN UID,Bus ID,Gen ID,Unit Group,Unit Type,Category,Fuel,MW Inj,MVAR Inj,V Setpoint p.u.,...,Emissions N2O Lbs/MMBTU,Emissions CO Lbs/MMBTU,Emissions VOCs Lbs/MMBTU,Damping Ratio,Inertia MJ/MW,Base MVA,Transformer X p.u.,Unit X p.u.,Pump Load MW,Storage Roundtrip Efficiency
74,122_HYDRO_1,122,1,U50,HYDRO,Hydro,Hydro,50.0,-6.79,1.05,...,0.0,0.0,0.0,0,3.5,53.0,0.1,0.28,0,0
75,122_HYDRO_2,122,2,U50,HYDRO,Hydro,Hydro,50.0,-6.79,1.05,...,0.0,0.0,0.0,0,3.5,53.0,0.1,0.28,0,0
76,122_HYDRO_3,122,3,U50,HYDRO,Hydro,Hydro,50.0,-6.79,1.05,...,0.0,0.0,0.0,0,3.5,53.0,0.1,0.28,0,0
77,122_HYDRO_4,122,4,U50,HYDRO,Hydro,Hydro,50.0,-6.79,1.05,...,0.0,0.0,0.0,0,3.5,53.0,0.1,0.28,0,0
78,122_HYDRO_5,122,5,U50,HYDRO,Hydro,Hydro,50.0,-6.79,1.05,...,0.0,0.0,0.0,0,3.5,53.0,0.1,0.28,0,0


In [18]:
df_special

,GEN UID,Bus ID,Gen ID,Unit Group,Unit Type,Category,Fuel,MW Inj,MVAR Inj,V Setpoint p.u.,...,Emissions N2O Lbs/MMBTU,Emissions CO Lbs/MMBTU,Emissions VOCs Lbs/MMBTU,Damping Ratio,Inertia MJ/MW,Base MVA,Transformer X p.u.,Unit X p.u.,Pump Load MW,Storage Roundtrip Efficiency
73,121_NUCLEAR_1,121,1,U400,NUCLEAR,Nuclear,Nuclear,400.0,-21.87,1.05,...,0.0,0.0,0.0,0,5.0,471.0,0.15,0.4,0,0
116,212_CSP_1,212,1,CSP,CSP,CSP,Solar,0.0,0.00,1.00,...,0.0,0.0,0.0,0,0.0,200.0,0.00,0.0,0,0


## Calculate cost for the start up with different suitations

In [19]:
for idx, row in df_fossil.iterrows():
    fuel_price = row['Fuel Price $/MMBTU']
    cold_start_energy = row['Start Heat Cold MBTU']
    warm_start_energy = row['Start Heat Warm MBTU']
    hot_start_energy = row['Start Heat Hot MBTU']
    cold_start_cost = fuel_price * cold_start_energy
    warm_start_cost = fuel_price * warm_start_energy
    hot_start_cost =  fuel_price * hot_start_energy

In [20]:
df_bus.columns.tolist()

['Bus ID',
 'Bus Name',
 'BaseKV',
 'Bus Type',
 'MW Load',
 'MVAR Load',
 'V Mag',
 'V Angle',
 'MW Shunt G',
 'MVAR Shunt B',
 'Area',
 'Sub Area',
 'Zone',
 'lat',
 'lng']

In [21]:
# generate a dictionary contains fossil generator information
fossil_dict = {}
for idx, row in df_fossil.iterrows():
    gen_name = row['GEN UID']
    gen_dict = {}
    gen_dict['name'] = gen_name
    bus_id= row['Bus ID']
    gen_dict['bus_name'] = df_bus[df_bus["Bus ID"]==bus_id]["Bus Name"].to_list()[0]
    gen_dict['gen_type'] = row['Unit Type']
    gen_dict['max_p'] = row['PMax MW']
    gen_dict['min_p'] = row['PMin MW']
    gen_dict['ramp'] = row['Ramp Rate MW/Min'] * 60 # ramp rate should be MW/hr
    gen_dict['fuel_p'] = row['Fuel Price $/MMBTU']
    gen_dict['min_down_time'] = int(np.round(row['Min Down Time Hr'], 0)) # should be rounded to an integer
    gen_dict['min_up_time'] = int(np.round(row['Min Up Time Hr'], 0)) # should be rounded to an integer
    gen_dict['start_up_time_hot'] = int(np.round(row['Start Time Hot Hr'], 0))
    gen_dict['start_up_time_warm'] = int(np.round(row['Start Time Warm Hr'], 0))
    gen_dict['start_up_time_cold'] = int(np.round(row['Start Time Cold Hr'], 0))
    gen_dict['start_heat_hot'] = row['Start Heat Hot MBTU']
    gen_dict['start_heat_warm'] = row['Start Heat Warm MBTU']
    gen_dict['start_heat_cold'] = row['Start Heat Cold MBTU']
    gen_dict['cost_curve'] = gen_param_dict[gen_name]
    
    fossil_dict[gen_name] = gen_dict

In [22]:
# generate a dictionary contains renewable generator information
renew_dict = {}
for idx, row in df_renew.iterrows():
    gen_name = row['GEN UID']
    gen_dict = {}
    gen_dict['name'] = gen_name
    gen_dict['gen_type'] = row['Unit Type']
    bus_id= row['Bus ID']
    gen_dict['bus_name'] = df_bus[df_bus["Bus ID"]==bus_id]["Bus Name"].to_list()[0]
    gen_dict['max_p'] = row['PMax MW']
#     gen_dict['min_p'] = row['PMin MW']
#     gen_dict['ramp'] = row['Ramp Rate MW/Min']
#     gen_dict['min_down_time'] = row['Min Down Time Hr']
#     gen_dict['min_up_time'] = row['Min Up Time Hr']
#     gen_dict['start_up_time_hot'] = row['Start Time Hot Hr']
#     gen_dict['start_up_time_cold'] = row['Start Time Cold Hr']
#     gen_dict['start_up_time_cold'] = row['Start Time Warm Hr']
    gen_dict['cost_curve'] = {'slope': 0, 'intercept': 0} # for the renewable generators, the operation cost is 0.
    
    renew_dict[gen_name] = gen_dict

## Generator startup type check

In [16]:
# check the generator types to decide its startup types

# def gen_startup_cost(type_gen_dict, gen_name):
#     min_down_time = type_gen_dict[gen_name]['min_down_time']
#     hot_time = type_gen_dict[gen_name]['start_up_time_hot']
#     warm_time = type_gen_dict[gen_name]['start_up_time_warm']
#     cold_time = type_gen_dict[gen_name]['start_up_time_cold']
#     fuel_p = type_gen_dict[gen_name]['fuel_p']
#     start_heat_hot = type_gen_dict[gen_name]['start_heat_hot']
#     start_heat_warm = type_gen_dict[gen_name]['start_heat_warm']
#     start_heat_cold = type_gen_dict[gen_name]['start_heat_cold']
    
#     start_up_cost_hot = start_heat_hot*fuel_p
#     start_up_cost_warm = start_heat_warm*fuel_p
#     start_up_cost_cold = start_heat_cold*fuel_p
#     start_up_cost = {'hot': start_up_cost_hot, 'warm': start_up_cost_warm, 'cold': start_up_cost_cold}
    
#     return start_up_cost

## Save the gen_dict.json

In [25]:
all_gen_dict = {}
all_gen_dict['fossil'] = fossil_dict
all_gen_dict['renew'] = renew_dict
# all_gen_dict['special'] = special_dict

gen_dict_path = os.path.join(os.getcwd(), '..', 'Data', 'gen_dict.json')

with open(gen_dict_path, "w") as f:
     json.dump(all_gen_dict, f)
     print("Successfully saved generator parameters to json files")

Successfully saved generator parameters to json files


# ERCOT 123 Bus

## Read gen.csv

In [21]:
# path for the generators

csv_path = os.path.join(os.getcwd(), '..', '..', 'idaes-gtep', 'gtep', 'data', '123_Bus_Coal', 'gen.csv')
df = read_gmlc_gen(csv_path)

# # path for the cost curves of fossil generators
# json_path = os.path.join(os.getcwd(), '..', 'Data', 'rtsgmle_gen_linear_cost_curve.json')
# with open(json_path, 'rb') as f:
#     gen_param_dict = json.load(f)

# path for the buses
bus_path = os.path.join(os.getcwd(), '..', '..', 'idaes-gtep', 'gtep', 'data', '123_Bus_Coal', 'bus.csv')
df_bus = pd.read_csv(bus_path)

In [22]:
# extract the CT, CC, STEAM, WIND, PV units
target_fossil_gen = []
target_renew_gen = []
target_special_gen = []
for idx, row in df.iterrows():
    if row['Unit Type'] in ['CT', 'COAL']:
        target_fossil_gen.append(row)
    if row['Unit Type'] in ['PV', 'WIND', 'HYDRO']:
        target_renew_gen.append(row)
    if row['Unit Type'] in ['NUC']:
        target_special_gen.append(row)
        
df_fossil = pd.DataFrame(target_fossil_gen)
df_renew = pd.DataFrame(target_renew_gen)
df_special = pd.DataFrame(target_special_gen)

In [23]:
df_fossil.columns

Index(['GEN UID', 'Bus ID', 'Unit Type', 'Fuel', 'MW Inj', 'MVAR Inj',
       'V Setpoint p.u.', 'PMax MW', 'PMin MW', 'QMax MVAR', 'QMin MVAR',
       'Min Down Time Hr', 'Min Up Time Hr', 'Ramp Rate MW/Min',
       'Start Time Cold Hr', 'Start Time Warm Hr', 'Start Time Hot Hr',
       'Start Heat Cold MBTU', 'Start Heat Warm MBTU', 'Start Heat Hot MBTU',
       'Non Fuel Start Cost $', 'Fuel Price $/MMBTU', 'Output_pct_0',
       'Output_pct_1', 'Output_pct_2', 'Output_pct_3', 'Output_pct_4',
       'HR_avg_0', 'HR_incr_1', 'HR_incr_2', 'HR_incr_3', 'HR_incr_4',
       'capex1', 'capex2', 'capex3', 'fuel_cost1', 'fuel_cost2', 'fuel_cost3',
       'fixed_ops1', 'fixed_ops2', 'fixed_ops3', 'var_ops1', 'var_ops2',
       'var_ops3'],
      dtype='object')

In [24]:
# generate a dictionary contains fossil generator information
fossil_dict = {}
for idx, row in df_fossil.iterrows():
    gen_name = row['GEN UID'] + '_' + row['Unit Type']
    gen_dict = {}
    gen_dict['name'] = gen_name
    bus_id= row['Bus ID']
    gen_dict['bus_name'] = row['Bus ID']
    gen_dict['gen_type'] = row['Unit Type']
    gen_dict['max_p'] = row['PMax MW']
    gen_dict['min_p'] = row['PMin MW']
    gen_dict['ramp'] = row['PMax MW'] # no ramp rate, set to 100%/hr
#     gen_dict['fuel_p'] = row['Fuel Price $/MMBTU']
    gen_dict['min_down_time'] = int(np.round(row['Min Down Time Hr'], 0)) # should be rounded to an integer
    gen_dict['min_up_time'] = int(np.round(row['Min Up Time Hr'], 0)) # should be rounded to an integer
#     gen_dict['start_up_time_hot'] = int(np.round(row['Start Time Hot Hr'], 0))
#     gen_dict['start_up_time_warm'] = int(np.round(row['Start Time Warm Hr'], 0))
#     gen_dict['start_up_time_cold'] = int(np.round(row['Start Time Cold Hr'], 0))
#     gen_dict['start_heat_hot'] = row['Start Heat Hot MBTU']
#     gen_dict['start_heat_warm'] = row['Start Heat Warm MBTU']
#     gen_dict['start_heat_cold'] = row['Start Heat Cold MBTU']
#     gen_dict['cost_curve'] = gen_param_dict[gen_name]
    
    fossil_dict[gen_name] = gen_dict
    
fossil_dict.keys()

dict_keys(['2_CT', '3_CT', '4_CT', '5_CT', '6_COAL', '7_CT', '8_CT', '10_CT', '11_CT', '12_CT', '13_CT', '14_CT', '15_CT', '16_CT', '18_CT', '25_CT', '26_COAL', '27_CT', '28_CT', '44_CT', '45_CT', '47_CT', '48_CT', '49_CT', '50_CT', '51_CT', '54_CT', '60_CT', '61_CT', '62_CT', '66_CT', '70_CT', '71_CT', '72_CT', '73_CT', '75_CT', '76_CT', '85_CT', '94_CT', '98_CT', '100_CT', '105_CT', '107_CT', '108_CT', '112_CT', '114_CT', '118_CT', '121_CT', '122_CT', '124_CT', '129_CT', '130_CT', '137_CT', '138_CT', '144_CT', '147_CT', '148_CT', '149_CT', '150_CT', '151_CT', '155_CT', '156_CT', '165_COAL', '166_CT', '171_CT', '174_CT', '175_COAL', '176_CT', '177_COAL', '178_CT', '179_CT', '180_CT', '181_COAL', '182_CT', '183_CT', '184_CT', '185_COAL', '186_CT', '187_COAL', '189_CT', '193_CT', '195_CT', '197_CT', '201_CT', '202_CT', '203_CT', '207_CT', '208_CT', '209_CT', '211_CT', '212_CT', '213_CT', '215_CT', '216_CT', '220_CT', '222_CT', '223_CT', '224_CT', '226_CT', '227_CT', '228_CT', '229_CT', 

# Test Rolling Horizon PT codes

Need to checkout the branch of idaes-pse to "rolling_horizon"

## Test RHPTForecaster

In [1]:
from idaes.apps.grid_integration.forecaster import RHPTForecaster
import numpy as np

In [2]:
p = []
for i in range(10):
    p.append([i]*24)
price = np.array(p).reshape(-1)

scenario = 5
horizon = 36
planning_horizon = 24

In [3]:
forecaster = RHPTForecaster(price, scenario, horizon, planning_horizon)

pointer=0
forecaster._forecast_prices(pointer)

2025-07-02 15:48:13 [INFO] idaes.apps.grid_integration.forecaster: Number of periods from provided data is 10.
2025-07-02 15:48:13 [INFO] idaes.apps.grid_integration.forecaster: Number of scenarios is 5.
2025-07-02 15:48:13 [INFO] idaes.apps.grid_integration.forecaster: The length of the scenario is 36.


array([[5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5,
        5, 5, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6],
       [6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6,
        6, 6, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7],
       [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7,
        7, 7, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8],
       [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8,
        8, 8, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9],
       [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9,
        9, 9, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])

In [7]:
csv_path = "../Data/all_bus_lmp.csv"
df_lmp = pd.read_csv(csv_path)
lmp_arr = df_lmp['Abel_LMP'].to_numpy()
len(lmp_arr)

8784

In [8]:
forecaster = RHPTForecaster(lmp_arr, scenario, horizon, planning_horizon)

2025-07-02 15:48:25 [INFO] idaes.apps.grid_integration.forecaster: Number of periods from provided data is 366.
2025-07-02 15:48:25 [INFO] idaes.apps.grid_integration.forecaster: Number of scenarios is 5.
2025-07-02 15:48:25 [INFO] idaes.apps.grid_integration.forecaster: The length of the scenario is 36.


In [10]:
pointer = 7
lmp_forecasted = forecaster._forecast_prices(pointer)
for idx, l in enumerate(lmp_forecasted):
    actual_l = lmp_arr[(2+idx)*24: (2+idx)*24 + horizon]
    print(len(l), l- actual_l)

36 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
36 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
36 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
36 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
36 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


## Test Rolling Horizon PT Results

In [33]:
from Customized_Determinstic_PT.utils import read_rolling_horizon_params, check_optimal_status, calculate_total_profit,calculate_total_startup_shutdown
# from Rolling_horizon.fossil_rolling_horizon_PT_parameter import gen_dict
# from idaes.apps.grid_integration import RHPTForecaster

In [32]:
# read the results dictionary
json_path = os.path.join(os.getcwd(), "..", "Data", "test_366_101_STEAM_3_result.json")
res_dict = read_rolling_horizon_params(json_path)

### Check if all period problem converged to optimal

In [30]:
non_optimal = check_optimal_status(res_dict)
print(non_optimal)

[]


### Compare the profit from PCM, PT and RHPT

In [31]:
actual_profit = calculate_total_profit(res_dict)
ideal_profit = calculate_total_profit(res_dict, name="IdeaProfit")

print(f"PCM Profit: {17.63} M$")
print(f"Standard Profit: {19.11} M$.")
print(f"Ideal Profit: {np.round(ideal_profit/1e6, 2)} M$.")
print(f"Actual Profit: {np.round(actual_profit/1e6, 2)} M$.")

PCM Profit: 17.63 M$
Standard Profit: 19.11 M$.
Ideal Profit: 16.96 M$.
Actual Profit: 14.66 M$.


### Compare the ideal and actual profit from RHPT

In [35]:
print("ideal", "actual")
for key in res_dict.keys():
    print(f"{np.round(res_dict[key]['IdeaProfit'], 2)}.", f"{np.round(res_dict[key]['ActualProfit'], 2)}.")

ideal actual
58311.6. -17108.49.
65379.29. -6068.05.
65447.93. -7794.03.
57795.8. 4857.58.
21257.66. -13649.83.
955.13. 13388.41.
-8993.15. 227.29.
5655.14. -16904.34.
4212.17. -11601.11.
1261.73. -7614.73.
2828.2. 2419.0.
-2602.7. 147650.05.
23765.99. 109230.09.
48471.16. -121.24.
50798.08. -7618.5.
50693.34. -12265.73.
47995.34. -431.27.
18368.18. -6204.83.
1205.68. 36863.78.
27799.13. 139152.48.
71577.96. 173866.7.
109702.21. 70484.27.
124305.16. -14735.88.
122522.83. -3614.56.
77736.01. -12467.8.
44033.97. -5744.03.
12157.03. -5973.94.
-12891.03. -17960.14.
-13244.82. -16239.33.
-14909.38. -15781.61.
-15387.22. -16347.35.
-15416.64. -16324.57.
27185.66. 31340.47.
57726.14. 152326.66.
103812.94. 133701.94.
134741.67. 164089.44.
170365.86. -914.58.
122317.27. 80765.42.
119339.31. 17772.63.
79127.99. 33656.67.
59146.2. 256925.87.
77732.88. 198268.26.
117599.6. -3178.99.
100659.23. 25306.03.
102195.57. 8422.4.
97148.71. -2764.94.
45210.55. -2134.85.
5133.31. 347331.77.
75274.87. 191967

### Find out startup and shutdown time

In [37]:
for key in res_dict.keys():
    total_shutdown = sum(res_dict[key]["101_STEAM_3"]["OperationVariables_shutdown"]["1"].values())
    total_startup = sum(res_dict[key]["101_STEAM_3"]["OperationVariables_startup"]["1"].values())
    if total_shutdown > 0:
        print("shutdown", key)
    if total_startup > 0:
        print("startup", key)

startup period_0
shutdown period_5
startup period_6
shutdown period_18
startup period_19
shutdown period_26
shutdown period_27
startup period_27
shutdown period_28
startup period_28
shutdown period_29
startup period_29
shutdown period_30
startup period_30
shutdown period_31
startup period_31
startup period_32
shutdown period_107
startup period_108
shutdown period_147
startup period_148
shutdown period_264
startup period_265
shutdown period_284
shutdown period_285
startup period_285
shutdown period_286
startup period_286
shutdown period_287
startup period_287
shutdown period_288
startup period_288
shutdown period_289
startup period_289
shutdown period_290
startup period_290
shutdown period_291
startup period_291
shutdown period_292
startup period_292
startup period_293
shutdown period_304
startup period_305
shutdown period_322
startup period_323
shutdown period_332
shutdown period_333
startup period_333
shutdown period_334
startup period_334
shutdown period_335
startup period_335
shutdo

In [39]:
calculate_total_startup_shutdown(res_dict, gen_name="101_STEAM_3")

(29.0, 28.0)

### Verify

In [49]:
slope = 16.466, 
intercept = 331.4380855711627

lmp_path = os.path.join("..", "Data", "all_bus_lmp.csv")
df_lmp = pd.read_csv(lmp_path)
lmp_data = df_lmp[gen_dict["gen_101_STEAM_3"]["bus_name"]+"_LMP"].to_numpy()

forecaster = RHPTForecaster(price_signal=lmp_data,
                            scenario=5,
                            horizon=36,
                            planning_horizon=24)

elec_rev = 0
cost = 0
pointer = 0

for i in res_dict:
    power =  res_dict[i]['gen_101_STEAM_3']['OperationVariables_power']['1']
    op_mode = res_dict[i]['gen_101_STEAM_3']['OperationVariables_op_mode']['1']
    power_arr = np.array([v for v in power.values()])
    op_mode_arr = np.array([v for v in op_mode.values()])
    lmp = forecaster.fetch_original_signal(pointer)
    cost += sum(power_arr * slope + op_mode_arr * intercept)
    elec_rev += sum(lmp*power_arr)
    pointer += 1
    
cost += 29*5284.8*2.11399

elec_rev, cost, elec_rev-cost

2025-08-15 00:26:07 [INFO] idaes.apps.grid_integration.forecaster: Number of periods from provided data is 366.
2025-08-15 00:26:07 [INFO] idaes.apps.grid_integration.forecaster: Number of scenarios is 5.
2025-08-15 00:26:07 [INFO] idaes.apps.grid_integration.forecaster: The length of the scenario is 36.


(26816963.368243992, 11831442.209015777, 14985521.159228215)

## Using new LMP

### PCM

In [32]:
df_lmp = pd.read_csv("Bus_LMP.csv")
df_dispatch = pd.read_csv("Generator_Dispatch.csv")

state_arr = df_dispatch["101_STEAM_3_Unit State"].to_numpy()

int_state_arr = []
for i in state_arr:
    if i:
        int_state_arr.append(1)
    else:
        int_state_arr.append(0)
int_state_arr = np.array(int_state_arr)

startups = 0
for i in range(1, len(state_arr)):
    if int(state_arr[i]) - int(state_arr[i-1]) == 1:
        startups += 1
        
dispatch_arr = df_dispatch["101_STEAM_3_Dispatch"].to_numpy()

costs = dispatch_arr * 16.466 + 331.4381*int_state_arr
sum(costs) + 2.11399*5284.8*6

# sum(df_lmp["Abel_LMP"].to_numpy() * dispatch_arr)

12467569.950188339

### Determinstic PT (fixed dispatch)

In [33]:
js_path = "../Data/det_fossil_PT_fixed_dispatch_results.json"

with open(js_path, "r") as f:
    res_dict = json.load(f)

startup = 0
rev = 0
vom = 0
for i in range(1, 366*24):
    startup += res_dict[str(i)]['startup']
    rev += res_dict[str(i)]['rev']
    vom += res_dict[str(i)]['vom']
    
startup, rev, vom + 2.11399*5284.8*6

(7.0, 29073470.33426585, 12465986.972160637)

### Determinstic PT (unfixed dispatch)

In [34]:
js_path = "../Data/det_fossil_PT_unfixed_dispatch_results.json"

with open(js_path, "r") as f:
    res_dict = json.load(f)

startup = 0
rev = 0
vom = 0
for i in range(1, 366*24):
    startup += res_dict[str(i)]['startup']
    rev += res_dict[str(i)]['rev']
    vom += res_dict[str(i)]['vom']
    
startup, rev, vom + 2.11399*5284.8*(startup-1)

(58.0, 28308104.85466783, 10165112.741302129)